In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import * 
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("sample_app") \
    .config("spark.sql.warehouse.dir",
            "s3a://netapp-data-sample/spark_warehouse/") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# S3 bucket details, access key, secret keys should be in environmental variable
# For s3 access
# load_dotenv("../.env")
# # load_dotenv(dotenv_path=env_path)

# print(os.getenv("AWS_ACCESS_KEY_ID"))

# credentials = {
#     "access_key": os.getenv("DB_HOST"),
#     "secret_key": os.getenv("DB_USER"),
# }

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
# spark._jsc.hadoopConfiguration().set(
#     "fs.s3a.access.key", credentials.get('AWS_ACCESS_KEY_ID'))
# spark._jsc.hadoopConfiguration().set(
#     "fs.s3a.secret.key", credentials.get('AWS_SECRET_ACCESS_KEY'))

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.access.key", '')
spark._jsc.hadoopConfiguration().set(
    "fs.s3a.secret.key", '')
spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint", "s3.amazonaws.com")

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint", "s3.amazonaws.com")

spark.sparkContext.setLogLevel("WARN")

from pyspark.sql import functions as F

#### Use of multiple spark session
#### 1. To test your logic in other session
#### 2. globalTempView will be available for multiple spark session

In [142]:
spark_session2 = spark.newSession()
print("session1", spark, "\nsession2", spark_session2)

session1 <pyspark.sql.session.SparkSession object at 0x792f45334c90> 
session2 <pyspark.sql.session.SparkSession object at 0x792fbc100f90>


#### Defining Schema
1. Schema DDL
2. structType and nested schema

In [4]:
from datetime import date

data = [(1, "a", date(2020, 1, 1))]
schema = 'id int, name string, dob date'
df = spark.createDataFrame(data, schema)
df.show()

+---+----+----------+
| id|name|       dob|
+---+----+----------+
|  1|   a|2020-01-01|
+---+----+----------+



In [317]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

schema = StructType([StructField("id", IntegerType()),
                    StructField("name", StringType(), True),
                    StructField("dob", DateType(), True),
                   StructField("address", StructType([StructField("city", StringType(), True),
                                                     StructField("zip", IntegerType(), True)]))])
data = [(1, "abc", date(2020, 1, 1), ("Blore", 560064))]
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+---+----+----------+---------------+
| id|name|       dob|        address|
+---+----+----------+---------------+
|  1| abc|2020-01-01|{Blore, 560064}|
+---+----+----------+---------------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: integer (nullable = true)



#### data cleaning and data validation
1. Don't infer schema, create schema manually
2. Rename columns if required
3. Add new column date_ingested(if required)
4. Check for duplicate rows (Or check for Primary key) and delete them
5. Delete null rows or fill them with something or derive them if possible according to requirement
6. Check for RANGE validation (for eg. revenue > 0)
7. Volume anomaly validation - Check if data volume is very high/low compared to usual data size
8. Write the cleaned data to CLEANED folder for further process/transformation

#### Read and Write to S3 and JDBC
#### Files Write modes: APPEND, OVERWRITE, ERROR, IGNORE

In [170]:
s3_read_df = spark.read.format("parquet").option("header", True).option("inferSchema", True).option("samplingRation", 0.5).load("s3a://dbt-snow-bucket/emp")
# It’s not recommended to infer the schema 
# •	as spark has to scan data to determine datatypes, so it is time consuming
# •	it might infer incorrectly (eg. order_date)
# If you still want to inter schema, we can do using sample data
# 0.5 => 50%  0.01 => 1%

s3_read_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- promoted: boolean (nullable = true)
 |-- state: string (nullable = true)
 |-- date_modified: timestamp (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = true)



In [145]:
s3_read_df.write.format("parquet").mode("append").save("s3a://dbt-snow-bucket/emp_save")  # Another part-0000 file will be added in same path
# s3_read_df.write.format("parquet").mode("overwrite").save("s3a://dbt-snow-bucket/emp_save")  # Overwrites files
# s3_read_df.write.format("parquet").mode("error").save("s3a://dbt-snow-bucket/emp_save")  # PATH_ALREADY_EXISTS error
# s3_read_df.write.format("parquet").mode("ignore").save("s3a://dbt-snow-bucket/emp_save")  # Error will not be thrown, data will be written only if foler is empty

In [161]:
# Read from DB increamentally, in parallel using fetchsize, upperbound and lowerbound
import builtins

jdbc_url = "jdbc:mysql://mysql_container:3306/employees" # JDBC connection URL, if DB is installed in someother server then jdbc:mysql://192.168.1.100:3306/company_db
user = 'root'
password = 'root'
driver = 'com.mysql.cj.jdbc.Driver' # Is a software library which allows java-based apps including spark to communicate with database
fetchsize = 1000
table = 'src_emp'
partition_column = 'emp_id'
num_partitions=10

# fetchsize=1000 num_partitions=10 👉 Means:
# Spark creates 10 parallel tasks (partitions)
# Each task opens 1 JDBC connection
# Each connection fetches data in chunks of 1000 rows per round trip
def base_reader():
    return (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", user)
        .option("password", password)
        .option("driver", driver)
        .option("fetchsize", fetchsize)
    )
        
# Get the upper and lower bound
bounds_query = f"(SELECT MIN({partition_column}) AS min_val,MAX({partition_column}) AS max_val,COUNT(*) AS total_count 
    FROM {table}) WHERE date_modified > '{last_read}') AS bounds"
bounds = (
    base_reader()
    .option("dbtable", bounds_query)
    .load()
    .first()
)

lower = bounds["min_val"]
upper = bounds["max_val"]
total = bounds["total_count"]

if total < num_partitions:
    num_partitions = builtins.max(1, int(total))
        
new_df = base_reader()\
    .option("dbtable", f"""
        (SELECT * FROM {table} 
         WHERE date_modified > '{last_read}') AS inc """)\
    .option("partitionColumn", partition_column)\
    .option("lowerBound", lower)\
    .option("upperBound", upper)\
    .option("numPartitions", num_partitions)\
    .load()
new_df.show()

+------+----------+------+--------+---------+-------------------+
|emp_id|      name|salary|promoted|    state|      date_modified|
+------+----------+------+--------+---------+-------------------+
|     1|Ravi Kumar| 55000|   false|Karnataka|2026-04-06 00:00:00|
|     9|     Hegde|255000|    true|Karnataka|2026-04-06 00:00:00|
+------+----------+------+--------+---------+-------------------+



In [172]:
new_df = new_df.repartition(8)  # Is like option("num_partitions", 8)

new_df.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "write_emp")\
    .option("user", user)\
    .option("password", password)\
    .option("driver", driver)\
    .option("batchsize", 5000)\
    .mode("append")\
    .option("truncate", False)\
    .option("isolationLevel", "READ_COMMITTED").save()

# | Use case                  | isolationLevel    |
# | ------------------------- | ----------------- |
# | General ETL               | `READ_COMMITTED`  |
# | High consistency required | `REPEATABLE_READ` |
# | Rare strict correctness   | `SERIALIZABLE`    |


##### Simple aggregation: n inputs => Output row 1
##### Grouping: n group => n output row
##### Window Functions - ROW_NUM(), RANK(), DENSE_RANK(), LEAD, LAG, Running total

In [ ]:
# CREATE TABLE sales_data (
#     id INT PRIMARY KEY,
#     salesperson VARCHAR(50),
#     region VARCHAR(50),
#     sale_date DATE,
#     amount INT
# );

# INSERT INTO sales_data (id, salesperson, region, sale_date, amount) VALUES
# (1, 'Amit', 'North', '2024-01-01', 500),
# (2, 'Amit', 'North', '2024-01-05', 700),
# (3, 'Amit', 'North', '2024-01-10', 700),
# (4, 'Ravi', 'South', '2024-01-02', 400),
# (5, 'Ravi', 'South', '2024-01-06', 600),
# (6, 'Ravi', 'South', '2024-01-11', 600),
# (7, 'Neha', 'West', '2024-01-03', 300),
# (8, 'Neha', 'West', '2024-01-07', 800),
# (9, 'Neha', 'West', '2024-01-12', 800);
# INSERT INTO sales_data (id, salesperson, region, sale_date, amount) VALUES
# (10, 'Amit', 'North', '2024-01-15', 900),
# (11, 'Amit', 'North', '2024-01-20', 300),
# (12, 'Amit', 'North', '2024-01-25', 400),
# (13, 'Ravi', 'South', '2024-01-15', 500),
# (14, 'Ravi', 'South', '2024-01-20', 700),
# (15, 'Ravi', 'South', '2024-01-25', 200),
# (16, 'Neha', 'West', '2024-01-15', 600),
# (17, 'Neha', 'West', '2024-01-20', 100),
# (18, 'Neha', 'West', '2024-01-25', 900);

In [2]:
jdbc_url = "jdbc:mysql://mysql_container:3306/employees"
input_df = spark.read.format("jdbc").option("url", jdbc_url)\
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", f"""
        (SELECT * FROM sales_data) AS inc """)\
    .option("user", "root") \
    .option("password", "root").load()
input_df.show(5)

+---+-----------+------+----------+------+
| id|salesperson|region| sale_date|amount|
+---+-----------+------+----------+------+
|  1|       Amit| North|2024-01-01|   500|
|  2|       Amit| North|2024-01-05|   700|
|  3|       Amit| North|2024-01-10|   700|
|  4|       Ravi| South|2024-01-02|   400|
|  5|       Ravi| South|2024-01-06|   600|
+---+-----------+------+----------+------+
only showing top 5 rows



In [186]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.amount)
output1_df = input_df.withColumn("row_number", row_number().over(window)).withColumn("rank", rank().over(window)).withColumn("dense_rank", dense_rank().over(window))
output1_df.select("salesperson", "amount", "row_number", "rank", "dense_rank").show()

+-----------+------+----------+----+----------+
|salesperson|amount|row_number|rank|dense_rank|
+-----------+------+----------+----+----------+
|       NULL|   900|         1|   1|         1|
|       Amit|   300|         1|   1|         1|
|       Amit|   400|         2|   2|         2|
|       Amit|   500|         3|   3|         3|
|       Amit|   700|         4|   4|         4|
|       Amit|   700|         5|   4|         4|
|       Amit|   900|         6|   6|         5|
|       Neha|   100|         1|   1|         1|
|       Neha|   300|         2|   2|         2|
|       Neha|   600|         3|   3|         3|
|       Neha|   800|         4|   4|         4|
|       Neha|   800|         5|   4|         4|
|       Neha|   900|         6|   6|         5|
|       Ravi|   200|         1|   1|         1|
|       Ravi|   400|         2|   2|         2|
|       Ravi|   500|         3|   3|         3|
|       Ravi|   600|         4|   4|         4|
|       Ravi|   600|         5|   4|    

In [187]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date)
output_df2 = input_df.withColumn("lead", lead(input_df.amount).over(window)).withColumn("lag", lag(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "lead", "lag").show()

+-----------+----------+------+----+----+
|salesperson| sale_date|amount|lead| lag|
+-----------+----------+------+----+----+
|       NULL|2024-01-15|   900|NULL|NULL|
|       Amit|2024-01-01|   500| 700|NULL|
|       Amit|2024-01-05|   700| 700| 500|
|       Amit|2024-01-10|   700| 900| 700|
|       Amit|2024-01-15|   900| 300| 700|
|       Amit|2024-01-20|   300| 400| 900|
|       Amit|2024-01-25|   400|NULL| 300|
|       Neha|2024-01-03|   300| 800|NULL|
|       Neha|2024-01-07|   800| 800| 300|
|       Neha|2024-01-12|   800| 600| 800|
|       Neha|2024-01-15|   600| 100| 800|
|       Neha|2024-01-20|   100| 900| 600|
|       Neha|2024-01-25|   900|NULL| 100|
|       Ravi|2024-01-02|   400| 600|NULL|
|       Ravi|2024-01-06|   600| 600| 400|
|       Ravi|2024-01-11|   600| 500| 600|
|       Ravi|2024-01-15|   500| 700| 600|
|       Ravi|2024-01-20|   700| 200| 500|
|       Ravi|2024-01-25|   200|NULL| 700|
+-----------+----------+------+----+----+



In [188]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date).rowsBetween(Window.unboundedPreceding, Window.currentRow)
output_df2 = input_df.withColumn("sum", sum(input_df.amount).over(window)).withColumn("avg", avg(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "sum", "avg").show()

+-----------+----------+------+----+-----------------+
|salesperson| sale_date|amount| sum|              avg|
+-----------+----------+------+----+-----------------+
|       NULL|2024-01-15|   900| 900|            900.0|
|       Amit|2024-01-01|   500| 500|            500.0|
|       Amit|2024-01-05|   700|1200|            600.0|
|       Amit|2024-01-10|   700|1900|633.3333333333334|
|       Amit|2024-01-15|   900|2800|            700.0|
|       Amit|2024-01-20|   300|3100|            620.0|
|       Amit|2024-01-25|   400|3500|583.3333333333334|
|       Neha|2024-01-03|   300| 300|            300.0|
|       Neha|2024-01-07|   800|1100|            550.0|
|       Neha|2024-01-12|   800|1900|633.3333333333334|
|       Neha|2024-01-15|   600|2500|            625.0|
|       Neha|2024-01-20|   100|2600|            520.0|
|       Neha|2024-01-25|   900|3500|583.3333333333334|
|       Ravi|2024-01-02|   400| 400|            400.0|
|       Ravi|2024-01-06|   600|1000|            500.0|
|       Ra

In [189]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date).rowsBetween(-2, Window.currentRow)  # Sum of last 3 rows 
output_df2 = input_df.withColumn("sum", sum(input_df.amount).over(window)).withColumn("avg", avg(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "sum", "avg").show()

+-----------+----------+------+----+-----------------+
|salesperson| sale_date|amount| sum|              avg|
+-----------+----------+------+----+-----------------+
|       NULL|2024-01-15|   900| 900|            900.0|
|       Amit|2024-01-01|   500| 500|            500.0|
|       Amit|2024-01-05|   700|1200|            600.0|
|       Amit|2024-01-10|   700|1900|633.3333333333334|
|       Amit|2024-01-15|   900|2300|766.6666666666666|
|       Amit|2024-01-20|   300|1900|633.3333333333334|
|       Amit|2024-01-25|   400|1600|533.3333333333334|
|       Neha|2024-01-03|   300| 300|            300.0|
|       Neha|2024-01-07|   800|1100|            550.0|
|       Neha|2024-01-12|   800|1900|633.3333333333334|
|       Neha|2024-01-15|   600|2200|733.3333333333334|
|       Neha|2024-01-20|   100|1500|            500.0|
|       Neha|2024-01-25|   900|1600|533.3333333333334|
|       Ravi|2024-01-02|   400| 400|            400.0|
|       Ravi|2024-01-06|   600|1000|            500.0|
|       Ra

#### count, count(col_name), countDistinct(col_name) excludes NULL
#### count(*) includes NULL

In [190]:
input_df.select(count("*"), count(input_df.salesperson), countDistinct(input_df.salesperson)).show()  
input_df.select(input_df.salesperson).distinct().show()

+--------+------------------+---------------------------+
|count(1)|count(salesperson)|count(DISTINCT salesperson)|
+--------+------------------+---------------------------+
|      19|                18|                          3|
+--------+------------------+---------------------------+

+-----------+
|salesperson|
+-----------+
|       Ravi|
|       Amit|
|       Neha|
|       NULL|
+-----------+



In [191]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date)
output_df3 = input_df.withColumn("sum", sum(input_df.amount).over(window))
output_df3.select("salesperson", "sale_date", "amount", "sum").show(10)

# Without orderBy, Aggregation column will be same for all rows in each group
window1 = Window.partitionBy(input_df.salesperson)
output_df3 = input_df.withColumn("sum", sum(input_df.amount).over(window1))
output_df3.select("salesperson", "sale_date", "amount", "sum").show(10)

+-----------+----------+------+----+
|salesperson| sale_date|amount| sum|
+-----------+----------+------+----+
|       NULL|2024-01-15|   900| 900|
|       Amit|2024-01-01|   500| 500|
|       Amit|2024-01-05|   700|1200|
|       Amit|2024-01-10|   700|1900|
|       Amit|2024-01-15|   900|2800|
|       Amit|2024-01-20|   300|3100|
|       Amit|2024-01-25|   400|3500|
|       Neha|2024-01-03|   300| 300|
|       Neha|2024-01-07|   800|1100|
|       Neha|2024-01-12|   800|1900|
+-----------+----------+------+----+
only showing top 10 rows

+-----------+----------+------+----+
|salesperson| sale_date|amount| sum|
+-----------+----------+------+----+
|       NULL|2024-01-15|   900| 900|
|       Amit|2024-01-01|   500|3500|
|       Amit|2024-01-05|   700|3500|
|       Amit|2024-01-10|   700|3500|
|       Amit|2024-01-15|   900|3500|
|       Amit|2024-01-20|   300|3500|
|       Amit|2024-01-25|   400|3500|
|       Neha|2024-01-03|   300|3500|
|       Neha|2024-01-07|   800|3500|
|       Neha

#### Some Transformations
* withColumn
* withColumnRenamed
* drop
* dropDuplicates
* case-when
* filterpivot
* isin
* isnotin => ~(isin)
* isNull()
* isNotNull()
* collect_list
* collect_set
* split by multiple char
* explode
* explode_outer
* posexplode
* date_format
* array_contains

In [206]:
transformed_df = input_df.withColumn("withColumn", lit('sample')).withColumn("oldColumn", lit("addColumn"))
transformed_df = transformed_df.withColumnRenamed("oldColumn", "withColumnRenamed").withColumn("casting", to_timestamp("sale_date"))
transformed_df.filter(transformed_df.region == 'North')
transformed_df = transformed_df.withColumn("case_when", when(transformed_df.region=='North', 'N')\
                                           .when(transformed_df.region=='South', 'S')\
                                           .when(transformed_df.region=='West', 'W')\
                                           .otherwise('E'))
transformed_df = transformed_df.drop(transformed_df.salesperson)
distinct_drop_df = transformed_df.select(transformed_df.id, transformed_df.region, transformed_df.amount)
distinct_drop_df.distinct().orderBy(distinct_drop_df.id).show()
distinct_drop_df.dropDuplicates().orderBy(distinct_drop_df.id).show()  # All columns are compared
distinct_drop_df.dropDuplicates(['region', 'amount']).orderBy(distinct_drop_df.id).show()

# distinct                              | dropduplicates
# Gives distinct vales in all columns	| Can specify which column to consider


+----+------+------+
|  id|region|amount|
+----+------+------+
|   1| North|   500|
|   2| North|   700|
|   3| North|   700|
|   4| South|   400|
|   5| South|   600|
|   6| South|   600|
|   7|  West|   300|
|   8|  West|   800|
|   9|  West|   800|
|  10| North|   900|
|  11| North|   300|
|  12| North|   400|
|  13| South|   500|
|  14| South|   700|
|  15| South|   200|
|  16|  West|   600|
|  17|  West|   100|
|  18|  West|   900|
|9999| North|   900|
+----+------+------+

+---+------+------+
| id|region|amount|
+---+------+------+
|  1| North|   500|
|  2| North|   700|
|  4| South|   400|
|  5| South|   600|
|  7|  West|   300|
|  8|  West|   800|
| 10| North|   900|
| 11| North|   300|
| 12| North|   400|
| 13| South|   500|
| 14| South|   700|
| 15| South|   200|
| 16|  West|   600|
| 17|  West|   100|
| 18|  West|   900|
+---+------+------+



#### Pivot
* GroupBy: Which column should be vertical. That is rows
* Pivot: Which column should be horrizontal. That is columns
* Aggregation function: What agg to perform. That is value

In [218]:
pivot_df = transformed_df.select('region', 'sale_date', 'amount').groupBy(transformed_df.sale_date).pivot('region').agg(sum(transformed_df.amount))
pivot_df.show()
# Verify result
transformed_df.filter((transformed_df.sale_date == '2024-01-15') & (transformed_df.region == 'North')).show()

+----------+-----+-----+----+
| sale_date|North|South|West|
+----------+-----+-----+----+
|2024-01-07| NULL| NULL| 800|
|2024-01-11| NULL|  600|NULL|
|2024-01-02| NULL|  400|NULL|
|2024-01-05|  700| NULL|NULL|
|2024-01-15| 1800|  500| 600|
|2024-01-10|  700| NULL|NULL|
|2024-01-12| NULL| NULL| 800|
|2024-01-03| NULL| NULL| 300|
|2024-01-20|  300|  700| 100|
|2024-01-06| NULL|  600|NULL|
|2024-01-01|  500| NULL|NULL|
|2024-01-25|  400|  200| 900|
+----------+-----+-----+----+

+----+------+----------+------+----------+-----------------+-------------------+-----------------+---------+
|  id|region| sale_date|amount|withColumn|withColumnRenamed|            casting|withColumnRenamed|case_when|
+----+------+----------+------+----------+-----------------+-------------------+-----------------+---------+
|  10| North|2024-01-15|   900|    sample|        addColumn|2024-01-15 00:00:00|        addColumn|        N|
|9999| North|2024-01-15|   900|    sample|        addColumn|2024-01-15 00:00:00|   

In [220]:
# To optimize
region_list = ['North', 'South', 'West', 'East']
pivot_df = transformed_df.groupBy(transformed_df.sale_date).pivot('region', region_list).agg(sum(transformed_df.amount))
pivot_df.show()

+----------+-----+-----+----+----+
| sale_date|North|South|West|East|
+----------+-----+-----+----+----+
|2024-01-07| NULL| NULL| 800|NULL|
|2024-01-11| NULL|  600|NULL|NULL|
|2024-01-02| NULL|  400|NULL|NULL|
|2024-01-05|  700| NULL|NULL|NULL|
|2024-01-15| 1800|  500| 600|NULL|
|2024-01-10|  700| NULL|NULL|NULL|
|2024-01-12| NULL| NULL| 800|NULL|
|2024-01-03| NULL| NULL| 300|NULL|
|2024-01-20|  300|  700| 100|NULL|
|2024-01-06| NULL|  600|NULL|NULL|
|2024-01-01|  500| NULL|NULL|NULL|
|2024-01-25|  400|  200| 900|NULL|
+----------+-----+-----+----+----+



In [232]:
transformed_df.filter(transformed_df.region.isin(['South'])).select('region').distinct().show()
transformed_df.filter(~transformed_df.region.isin(['South'])).select('region').distinct().show()

pivot_df.filter(pivot_df.North.isNull()).select('North').distinct().show()
pivot_df.filter(pivot_df.North.isNotNull()).select('North').show()

+------+
|region|
+------+
| South|
+------+

+------+
|region|
+------+
|  West|
| North|
+------+

+-----+
|North|
+-----+
| NULL|
+-----+

+-----+
|North|
+-----+
|  700|
| 1800|
|  700|
|  300|
|  500|
|  400|
+-----+



In [238]:
transformed_df.groupBy(transformed_df.region)\
    .agg(collect_list(transformed_df.amount).alias("collect_list")).show(2, False)
transformed_df.groupBy(transformed_df.region)\
    .agg(collect_set(transformed_df.amount).alias("collect_list")).show(2, False)

+------+------------------------------+
|region|collect_list                  |
+------+------------------------------+
|South |[400, 600, 600, 500, 700, 200]|
|West  |[300, 800, 800, 600, 100, 900]|
+------+------------------------------+
only showing top 2 rows

+------+-------------------------+
|region|collect_list             |
+------+-------------------------+
|South |[700, 500, 600, 400, 200]|
|West  |[300, 800, 100, 900, 600]|
+------+-------------------------+
only showing top 2 rows



In [277]:
data = [(None, None), ("chaitra,hegde", 30), ("shiv-hegde", 2), ("shubh,g-hegde", 2), ("abc,gouda", 5)]
schema = "name string, age int"
df = spark.createDataFrame(data, schema)
df.show()
df.withColumn("explode", explode(split(col("name"), "[,-]"))).show()  # In single line
df = df.withColumn("split_name", split(col("name"), "[,-]"))  # Multiple splits
df.withColumn("explode", explode(col("split_name"))).show()  # Null re excluded
df.withColumn("explode_outer", explode_outer(col("split_name"))).show()  # Null re excluded
df.select("*", posexplode(col("split_name")).alias("pos", "posexplode")).show()  # WithColumn wont work here because posexplode returns 2 column - pos and value
df.withColumn("array_contains", array_contains(col("split_name"), "hegde")).show()

+-------------+----+
|         name| age|
+-------------+----+
|         NULL|NULL|
|chaitra,hegde|  30|
|   shiv-hegde|   2|
|shubh,g-hegde|   2|
|    abc,gouda|   5|
+-------------+----+

+-------------+---+-------+
|         name|age|explode|
+-------------+---+-------+
|chaitra,hegde| 30|chaitra|
|chaitra,hegde| 30|  hegde|
|   shiv-hegde|  2|   shiv|
|   shiv-hegde|  2|  hegde|
|shubh,g-hegde|  2|  shubh|
|shubh,g-hegde|  2|      g|
|shubh,g-hegde|  2|  hegde|
|    abc,gouda|  5|    abc|
|    abc,gouda|  5|  gouda|
+-------------+---+-------+

+-------------+---+-----------------+-------+
|         name|age|       split_name|explode|
+-------------+---+-----------------+-------+
|chaitra,hegde| 30| [chaitra, hegde]|chaitra|
|chaitra,hegde| 30| [chaitra, hegde]|  hegde|
|   shiv-hegde|  2|    [shiv, hegde]|   shiv|
|   shiv-hegde|  2|    [shiv, hegde]|  hegde|
|shubh,g-hegde|  2|[shubh, g, hegde]|  shubh|
|shubh,g-hegde|  2|[shubh, g, hegde]|      g|
|shubh,g-hegde|  2|[shubh, g, h

In [275]:
transformed_df.select(date_format(transformed_df.sale_date, "MMMM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MMM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "yyyy")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "yy")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "dd")).show(1)

+----------------------------+
|date_format(sale_date, MMMM)|
+----------------------------+
|                     January|
+----------------------------+
only showing top 1 row

+---------------------------+
|date_format(sale_date, MMM)|
+---------------------------+
|                        Jan|
+---------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, MM)|
+--------------------------+
|                        01|
+--------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, MM)|
+--------------------------+
|                        01|
+--------------------------+
only showing top 1 row

+----------------------------+
|date_format(sale_date, yyyy)|
+----------------------------+
|                        2024|
+----------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, yy)|
+--------------------------+
|                        24|
+-------------

#### Joins

In [301]:
data1 = [
    (1, "Chaitra", 2),
    (2, "Shiv", 2),
    (3, "Shubh", 3),
    (4, "Ram", 5),
    (5, "abc", 6)
]

data2 = [
    (1, "HR"),
    (2, "IT"),
    (3, "Finance"),
    (4, "Sales")
]

schema1 = ["emp_id", "name", "dept_id"]
schema2 = ["dept_id", "department"]

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

df1.show()
df2.show()

+------+-------+-------+
|emp_id|   name|dept_id|
+------+-------+-------+
|     1|Chaitra|      2|
|     2|   Shiv|      2|
|     3|  Shubh|      3|
|     4|    Ram|      5|
|     5|    abc|      6|
+------+-------+-------+

+-------+----------+
|dept_id|department|
+-------+----------+
|      1|        HR|
|      2|        IT|
|      3|   Finance|
|      4|     Sales|
+-------+----------+



In [302]:
df1.join(df2, on="dept_id", how="inner").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
+-------+------+-------+----------+



In [303]:
df1.join(df2, on="dept_id", how="left").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
|      5|     4|    Ram|      NULL|
|      6|     5|    abc|      NULL|
+-------+------+-------+----------+



In [304]:
df1.join(df2, on="dept_id", how="right").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      1|  NULL|   NULL|        HR|
|      2|     2|   Shiv|        IT|
|      2|     1|Chaitra|        IT|
|      3|     3|  Shubh|   Finance|
|      4|  NULL|   NULL|     Sales|
+-------+------+-------+----------+



In [305]:
df1.join(df2, on="dept_id", how="outer").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      1|  NULL|   NULL|        HR|
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
|      4|  NULL|   NULL|     Sales|
|      5|     4|    Ram|      NULL|
|      6|     5|    abc|      NULL|
+-------+------+-------+----------+



In [306]:
df1.join(df2, on="dept_id", how="left_semi").show()  # Same as inner but includes columns only from left df

+-------+------+-------+
|dept_id|emp_id|   name|
+-------+------+-------+
|      2|     1|Chaitra|
|      2|     2|   Shiv|
|      3|     3|  Shubh|
+-------+------+-------+



In [288]:
df1.join(df2, on="dept_id", how="left_anti").show()  # df1 - df2 select * from left_table where ID not in (select ID from right_table)

+-------+------+----+
|dept_id|emp_id|name|
+-------+------+----+
|      5|     4| Ram|
|      6|     5| abc|
+-------+------+----+



In [289]:
df1.crossJoin(df2).show()  # Cartesian product

+------+-------+-------+-------+----------+
|emp_id|   name|dept_id|dept_id|department|
+------+-------+-------+-------+----------+
|     1|Chaitra|      2|      1|        HR|
|     1|Chaitra|      2|      2|        IT|
|     1|Chaitra|      2|      3|   Finance|
|     1|Chaitra|      2|      4|     Sales|
|     2|   Shiv|      2|      1|        HR|
|     2|   Shiv|      2|      2|        IT|
|     2|   Shiv|      2|      3|   Finance|
|     2|   Shiv|      2|      4|     Sales|
|     3|  Shubh|      3|      1|        HR|
|     3|  Shubh|      3|      2|        IT|
|     3|  Shubh|      3|      3|   Finance|
|     3|  Shubh|      3|      4|     Sales|
|     4|    Ram|      5|      1|        HR|
|     4|    Ram|      5|      2|        IT|
|     4|    Ram|      5|      3|   Finance|
|     4|    Ram|      5|      4|     Sales|
|     5|    abc|      6|      1|        HR|
|     5|    abc|      6|      2|        IT|
|     5|    abc|      6|      3|   Finance|
|     5|    abc|      6|      4|

#### Union VS UnionByName

In [296]:
data1 = [
    (1, "Chaitra"),
    (2, "Shiv"),
    (3, "Shubh"),
]

data2 = [
    ("Shiv", 2),
    ("Shubh", 3),
    ("abc", 4),
]

schema1 = ["emp_id", "name"]
schema2 = ["name", "emp_id"]

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

df1.show()
df2.show()
df1.union(df2).show()
df1.unionByName(df2).show()

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
+------+-------+

+-----+------+
| name|emp_id|
+-----+------+
| Shiv|     2|
|Shubh|     3|
|  abc|     4|
+-----+------+

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
|  Shiv|      2|
| Shubh|      3|
|   abc|      4|
+------+-------+

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
|     2|   Shiv|
|     3|  Shubh|
|     4|    abc|
+------+-------+



In [299]:
df1.sort(['emp_id'], ascending=[0,1]).show()

+------+-------+
|emp_id|   name|
+------+-------+
|     3|  Shubh|
|     2|   Shiv|
|     1|Chaitra|
+------+-------+



#### Incremental Loading + Delta MERGE 

In [87]:
# CREATE TABLE src_emp (
#     emp_id INT PRIMARY KEY,
#     name VARCHAR(100) NOT NULL,
#     salary INT,
#     promoted BOOLEAN,
#     state VARCHAR(50),
#     date_modified TIMESTAMP
# );

# INSERT INTO src_emp (emp_id, name, salary, promoted, state, date_modified) VALUES
# (1, 'Ravi Kumar', 55000.00, 1, 'Karnataka', '2026-04-01'),
# (2, 'Anita Sharma', 62000.50, 0, 'Maharashtra', '2026-03-20'),
# (3, 'Suresh Reddy', 48000.75, 1, 'Telangana', '2024-01-01'),
# (4, 'Priya Nair', 71000.00, 1, 'Kerala', '2026-02-15'),
# (5, 'Amit Singh', 53000.25, 0, 'Delhi', '2022-06-01'),
# (6, 'Sneha Patil', 60000.00, 1, 'Maharashtra', '2026-04-05'),
# (7, 'Arjun Das', 45000.00, 0, 'West Bengal', '2026-03-30'),
# (8, 'Meera Iyer', 68000.90, 1, 'Tamil Nadu', '2026-01-10');

from delta.tables import DeltaTable
try:
    last_read_df = spark.read.format("delta").load("s3a://netapp-data-sample/emp")
    last_read = last_read_df.select(max(last_read_df.date_modified)).collect()[0][0]
    old_df = DeltaTable.forPath(spark, "s3a://netapp-data-sample/emp")
except Exception as e:
    print(e)
    last_read = '1900-01-01'
print(last_read)

[PATH_NOT_FOUND] Path does not exist: s3a://netapp-data-sample/emp.
1900-01-01


In [86]:
jdbc_url = "jdbc:mysql://mysql_container:3306/employees"
new_df = spark.read.format("jdbc").option("url", jdbc_url)\
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", f"""
        (SELECT * FROM src_emp 
         WHERE date_modified > '{last_read}') AS inc """)\
    .option("user", "root") \
    .option("password", "root").load()
new_df.show()

+------+------------+------+--------+-----------+-------------------+
|emp_id|        name|salary|promoted|      state|      date_modified|
+------+------------+------+--------+-----------+-------------------+
|     1|  Ravi Kumar| 55000|    true|  Karnataka|2026-04-01 00:00:00|
|     2|Anita Sharma| 62001|   false|Maharashtra|2026-03-20 00:00:00|
|     3|Suresh Reddy| 48001|    true|  Telangana|2024-01-01 00:00:00|
|     4|  Priya Nair| 71000|    true|     Kerala|2026-02-15 00:00:00|
|     5|  Amit Singh| 53000|   false|      Delhi|2022-06-01 00:00:00|
|     6| Sneha Patil| 60000|    true|Maharashtra|2026-04-05 00:00:00|
|     7|   Arjun Das| 45000|   false|West Bengal|2026-03-30 00:00:00|
|     8|  Meera Iyer| 68001|    true| Tamil Nadu|2026-01-10 00:00:00|
+------+------------+------+--------+-----------+-------------------+



In [25]:
new_df = new_df.withColumn("start_date", current_date())\
    .withColumn("end_date", lit(None).cast(DateType())) \
    .withColumn("is_current", lit(True))

if last_read == "1900-01-01":
    new_df.write.format("delta").mode("append").save("s3a://dbt-snow-bucket/emp")

else:
    old_df.alias("old").merge(
        new_df.alias("new"),
        "old.emp_id = new.emp_id AND old.is_current = true"
    ).whenMatchedUpdate(
        condition="old.promoted <> new.promoted",
        set={
            "end_date": "current_date()",
            "is_current": "false"
        }
    ).execute()

    # Insert new version of changed records
    new_records = new_df.withColumn("start_date", current_date()) \
                       .withColumn("end_date", lit(None)) \
                       .withColumn("is_current", lit(True))
    
    old_df.alias("old").merge(
        new_records.alias("new"),
        "old.emp_id = new.emp_id AND old.is_current = true"
    ).whenNotMatchedInsertAll().execute()


In [3]:
# After seconf load
last_read_df = spark.read.format("delta").load("s3a://dbt-snow-bucket/emp")
last_read_df.show()

+------+------------+------+--------+-----------+-------------------+----------+----------+----------+
|emp_id|        name|salary|promoted|      state|      date_modified|start_date|  end_date|is_current|
+------+------------+------+--------+-----------+-------------------+----------+----------+----------+
|     1|  Ravi Kumar| 55000|    true|  Karnataka|2026-04-01 00:00:00|2026-04-21|2026-04-21|     false|
|     6| Sneha Patil| 60000|    true|Maharashtra|2026-04-05 00:00:00|2026-04-21|      NULL|      true|
|     3|Suresh Reddy| 48001|    true|  Telangana|2024-01-01 00:00:00|2026-04-21|      NULL|      true|
|     5|  Amit Singh| 53000|   false|      Delhi|2022-06-01 00:00:00|2026-04-21|      NULL|      true|
|     4|  Priya Nair| 71000|    true|     Kerala|2026-02-15 00:00:00|2026-04-21|      NULL|      true|
|     8|  Meera Iyer| 68001|    true| Tamil Nadu|2026-01-10 00:00:00|2026-04-21|      NULL|      true|
|     7|   Arjun Das| 45000|   false|West Bengal|2026-03-30 00:00:00|2026

In [17]:
# After First load
last_read_df = spark.read.format("delta").load("s3a://dbt-snow-bucket/emp")
last_read_df.show()

+------+------------+------+--------+-----------+-------------------+----------+--------+----------+
|emp_id|        name|salary|promoted|      state|      date_modified|start_date|end_date|is_current|
+------+------------+------+--------+-----------+-------------------+----------+--------+----------+
|     1|  Ravi Kumar| 55000|    true|  Karnataka|2026-04-01 00:00:00|2026-04-21|    NULL|      true|
|     2|Anita Sharma| 62001|   false|Maharashtra|2026-03-20 00:00:00|2026-04-21|    NULL|      true|
|     3|Suresh Reddy| 48001|    true|  Telangana|2024-01-01 00:00:00|2026-04-21|    NULL|      true|
|     4|  Priya Nair| 71000|    true|     Kerala|2026-02-15 00:00:00|2026-04-21|    NULL|      true|
|     5|  Amit Singh| 53000|   false|      Delhi|2022-06-01 00:00:00|2026-04-21|    NULL|      true|
|     6| Sneha Patil| 60000|    true|Maharashtra|2026-04-05 00:00:00|2026-04-21|    NULL|      true|
|     7|   Arjun Das| 45000|   false|West Bengal|2026-03-30 00:00:00|2026-04-21|    NULL|  

In [13]:
Z order demo
data cleaning 
data validation
parity checks


```sql
SELECT user_id, COUNT(*) AS orders
FROM orders
WHERE status = 'completed'
GROUP BY user_id;

| Concept        | Description                     |
|----------------|---------------------------------|
| DBT            | Data transformation tool        |
| Model          | SQL file in DBT                 |
| Materialization| Table/View creation strategy    |

<details>
<summary><b>Click to expand</b></summary>

Your notes go here

- Point 1  
- Point 2  

</details>

#### Interview questions
##### Number of stages created

In [19]:
df1 = spark.read.format("csv").option("header", True).option("inferSchema", True).load("s3a://netapp-data-sample/raw/volume_20260102.csv")
df2 = spark.read.format("csv").option("header", True).option("inferSchema", True).load("s3a://netapp-data-sample/raw/volume_20260103.csv")
df1.printSchema()

root
 |-- volume_uuid: string (nullable = true)
 |-- volume_name: string (nullable = true)
 |-- cluster_uuid: string (nullable = true)
 |-- svm_uuid: string (nullable = true)
 |-- aggregate_name: string (nullable = true)
 |-- volume_type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- security_style: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- size_bytes: double (nullable = true)
 |-- used_bytes: double (nullable = true)
 |-- storage_efficiency_percent: double (nullable = true)
 |-- updated_at: timestamp (nullable = true)



In [20]:
df3 = df1.join(df2, df1.cluster_uuid == df2.cluster_uuid).select(df1.cluster_uuid)
df3.write.format("parquet").save("s3a://netapp-data-sample/int111111")

#### The 5 V’s of Big Data
*   **Volume:** Size of data.
*   **Variety:** Type of data.
*   **Value:** Benefits that an organization can get from data.
*   **Velocity:** Speed at which data is generated.
*   **Veracity:** Quality and trustworthiness of data.

#### What is Spark?
Spark is a cluster computing platform designed for handling large-scale data. We can install Spark on multiple machines, group them into a cluster, and assign them tasks to complete. It acts as a distributed computing engine that splits and distributes data across nodes to process it efficiently.

##### Spark Architecture

<img src="spark_archi.png" width="60%">

1. **Job Submission:** When we submit the code, the Cluster Manager starts the Driver program on one of the nodes/machines in the cluster.
2. **Task Breakdown:** The Driver breaks the code down into transformations, actions, stages, and tasks. It hands them over to the Cluster Manager and requests $n$ executor nodes.
3. **Execution:** The code is transferred to the designated executor nodes. The Executor nodes execute the tasks and either write the final output to a specified storage location or return the results back to the Driver depending on the code logic.

##### YARN-SPARK Workflow
1. **Job Submission:** The user submits the Spark job.
2. **Resource Manager Allocation:** The job goes to the Resource Manager (RM).
3. **Driver Initialization:** The Resource Manager contacts one of the worker nodes and spins up the Application Master (AM), which hosts the Spark Driver.
4. **Job Ownership:** The Application Master takes full responsibility for coordinating and completing the job.
5. **Data Locality Check:** The AM contacts the NameNode to check the metadata and find out exactly which DataNodes are storing the required file blocks.
6. **Resource Request & Execution:** Once the AM gets this data locality info, it requests specific worker nodes from the RM to complete the job right where the data lives. job

, json
e


##### Features/Benefits of Spark
1. In-memory computation
2. Lazy evaluation: All transformations will be stored in a logical plan and when we apply actions then only all those transformations will be executed
3. Fault tolerant: We can trace back all transformation
4. Distributed
5. Supports batch processing and streaming
6. Supports machine learning, Graphx, streaming, SQL
7. Supports many files formats like parquet, text, json

##### Job, Stage, Task, App

<img src="stg.png" width="10%">

*   **Task:** Single operation applied on single partition. *(Eg: if a df has 2 partitions and filter is applied on those 2 partitions, then there will be 2 tasks)*
*   **Stage:** Set of Tasks which can be run in parallel without any data shuffle.
*   **Job:** Set of stages, jobs are triggered by actions.
*   **App:** Main function.

**Key Formulas:**
*   Number of jobs = Number of actions
*   Number stages = Number of wide transformations + 1
*   Number of tasks = Number of partitions

**When are stages created?** 
Stages are created when there is a **data shuffle**.
*   *Example flow:* Read file $\rightarrow$ Apply filter $\rightarrow$ Apply groupBy $\rightarrow$ Save to file.
*   *Result:* **2 stages** will be created. 
    *   **Stage 1:** Read file and apply filter.
    *   **Stage 2:** Apply groupBy and save file.

<img src="executor.png" width="30%">

 **Storage memory:** memory used for caching
By and save file.



### Spark Submit Command

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --driver-memory 10g \
  --executor-memory 2g \
  --files config.yaml \
  --jars $PWD/mysql-connector.jar \
  --driver-class-path $PWD/mysql-connector.jar \
  $PWD/file.py
```

---

### Configurations Breakdown

#### `--deploy-mode`
*   **`cluster`**
    *   The Driver program runs on one of the worker nodes inside the cluster.
    *   Highly recommended for **production environments** because it prevents the submission machine from becoming a network bottleneck.
*   **`client`**
    *   The Driver program runs locally on the machine where you are submitting the job.
    *   The Driver runs outside the cluster framework.
    *   Ideal for **development, testing, and interactive debugging purposes**.

#### `--master`
*   **`local`**
    *   Runs Spark locally on a single machine without any parallelism (uses only 1 core).
*   **`local[*]`**
    *   Runs Spark locally, but automatically scales to utilize as many worker threads as there are logical CPU cores available on your system.
*   **`spark://host:port`**
    *   Connects the job to a Spark Standalone cluster manager. *(The default port is usually `7077`)*.
*   **`yarn`**
    *   Connects the job to a YARN resource manager cluster. Highly integrated with Hadoop ecosystems.
file.py

### Spark Context vs. SparkSession

| Feature / Concept | SparkContext | SparkSession |
| :--- | :--- | :--- |
| **Role** | Entry point for older versions (Spark 1.x) | Entry point for newer versions (Spark 2.x+) |
| **API Level** | Lower-level operations (RDDs) | Higher-level APIs (DataFrames, Datasets, Spark SQL) |
| **Scope** | Represents the entire Spark **Application** | Represents a specific **User Session** within that application |

*   **Core Responsibility:** The `SparkContext` is responsible for establishing a connection to the cluster manager and pushing your queries/commands down to the cluster executors.
*   **Dependency:** To utilize any core Spark APIs under the hood, a `SparkContext` must be initialized. In newer setups, the `SparkSession` automatically creates and wraps this context for you.

---

### Repartitioning vs. Coalesce

| Feature | Repartition | Coalesce |
| :--- | :--- | :--- |
| **Data Shuffle** | Triggers a **complete reshuffle** across nodes | Avoids full shuffles; **minimal data movement** |
| **Partition Size** | Creates partitions of roughly **equal size** | Can result in **unequal partition sizes** |
| **Scaling Capability**| Can **increase or decrease** partition count | Can **only decrease** partition count |
| **Resource Cost** | High performance overhead (heavy disk/network I/O) | Much faster and cheaper operation |

---

### Real-World Partitioning Scenarios

#### Scenario 1: When to use `Repartition` (To Increase Partitions)
*   **The Problem:** You have a **5 GB file** running on a **100-node cluster**. 
    *   By default (128 MB blocks), Spark creates: \(\frac{5 \times 1024 \text{ MB}}{128 \text{ MB}} = 40 \text{ partitions}\).
    *   This maps to 1 partition per node across 40 nodes, leaving **60 nodes completely idle**.
*   **The Solution:** Use `repartition()` to scale up the partition count, ensuring all 100 cluster nodes participate in parallel execution.

#### Scenario 2: When to use `Coalesce` (To Decrease Partitions)
*   **The Problem:** You start with a massive **1 TB dataset** on a **100-node cluster**, creating **8,000 partitions** (~ 80 partitions per node). 
    *   You apply a heavy `.filter()` operation that drops 90% of your data. 
    *   Now, you are stuck with 8,000 tiny or empty partitions, creating massive management overhead (the "small file problem").
*   **The Solution:** Use `coalesce()` to downscale the number of partitions cleanly without inducing a costly data reshuffle across your nodes.


### Broadcast Variables vs. Accumulators

| Feature | Broadcast Variables | Accumulators |
| :--- | :--- | :--- |
| **Primary Purpose** | Efficiently **sharing read-only data** across the cluster. | Efficiently **aggregating/writing metrics** across the cluster. |
| **Operation Type** | **Read-Only** by the executors. | **Write-Only** (Append/Add) by the executors; read-only by the Driver. |
| **Caching Mechanism**| Cached explicitly in the memory of **each Executor node** once, rather than shipped repeatedly with every single task. | Distributed counters shared among multiple executors that send updates back to the Driver. |
| **Common Use Cases** | Small lookup tables, machine learning model weights, and global configuration maps. | Counters (e.g., counting corrupted/null rows), tracking custom metrics, and debugging tasks. |

---

### Deep Dive & Implementation Context

#### Broadcast Variables
*   **When to use:** Use these when multiple tasks across multiple stages need to constantly refer to the same static dataset. 
*   **Performance Impact:** Instead of serializing and sending a dataset over the network for every single task, Spark sends it to the executor machine **exactly once**, drastically reducing network I/O.

#### Accumulators
*   **The DataFrame Paradigm:** When working with modern Spark DataFrames or Spark SQL, you should rely on native DataFrame transformations and aggregations (`.groupBy()`, `.sum()`, etc.), as Spark optimizes these automatically.
*   **The RDD/Custom Logic Paradigm:** Accumulators are specifically useful when you are forced to write **custom transformation logic** or low-level **RDD operations** (like a `.foreach()` loop or a MapPartitions operation) where standard SQL aggregations cannot easily hook into the row-by-row iteration.


##### Broadcast Joins & Variables
A broadcast join optimizes lookups by sending the smaller dataset to every node in the cluster, completely avoiding a costly data shuffle.

*   **Mechanism:** The smaller dataset is collected to the driver and sent to each worker node to be kept in memory locally.
*   **PySpark Syntax:**
    ```python
    from pyspark.sql.functions import broadcast
    
    # Broadcast join on DataFrames
    joined_df = large_df.join(broadcast(small_df), "join_key")
    ```

##### Spark Execution: Logical vs. Physical Plans

<img src="plans.png" width="50%">
When you execute a query, the **Catalyst Optimizer** processes it through four main planning phases:

1.  **Unresolved (Parsed) Logical Plan:** Generated immediately after a syntax and parsing check. The names of tables or columns are not yet validated.
2.  **Analyzed Logical Plan:** Generated after cross-referencing metadata catalogs to verify schema definitions, table names, and column data types.
3.  **Optimized Logical Plan:** Restructures logical operations (like filters, joins, and projections) using rule-based strategies.
    *   **Predicate Pushdown:** Moves `WHERE` or `.filter()` clauses as close to the data source as possible. This ensures data is filtered out *before* reading it into memory, significantly reducing I/O.
    *   **Filter Combining:** Merges multiple sequential filters into a single condition.
    *   **Column Projection Pruning:** Combines multiple selection statements so only the necessary columns are loaded.
4.  **Physical Plan:** The Catalyst Optimizer generates multiple valid physical execution strategies. It runs them through a **Cost-Based Optimizer (CBO)** to estimate resource and performance costs. Only the single most efficient physical plan is selected for final execution.
    *   *Decisions made here include:* Selecting join strategies (e.g., Broadcast Hash Join vs. Sort-Merge Join) and choosing aggregation methods (e.g., Hash Aggregation vs. Sort Aggregation).

##### Catalyst Optimizer Core Features
The Catalyst Optimizer is Spark SQL’s core optimization framework. It relies on a set of predefined rules that developers can optionally extend with custom optimization patterns.

*   Analyzes structural logical paths.
*   Applies both **rule-based** and **cost-based** code transformations.
*   Generates the final executable physical machine code.
*   Enables critical data optimizations out-of-the-box like **predicate pushdown**, **projection pruning** (ignoring unused columns), and **join re-ordering**.

##### Transformations: Narrow vs. Wide
Transformations represent the lineage of operations applied to your data.

| Metric | Narrow Transformations | Wide Transformations |
| :--- | :--- | :--- |
| **Data Shuffle** | **No data shuffling** occurs across the network. | Requires a **full data shuffle** across cluster nodes. |
| **Dependency** | Each partition in the child RDD/DataFrame depends on a single parent partition. | Partitions in the child RDD/DataFrame depend on data from multiple parent partitions. |
| **Examples** | `map()`, `filter()`, `flatMap()` | `reduceByKey()`, `groupByKey()`, `join()`, `repartition()` |

> 💡 **Tip:** Using `df.sortWithinPartitions("column_name")` acts as a narrow transformation. It sorts data strictly within its current local borders and successfully **avoids a global cluster shuffle**.

##### API Comparison: RDD vs. DataFrame vs. Dataset

| Feature / Attribute | RDD (Resilient Distributed Dataset) | DataFrame / Spark SQL | Dataset |
| :--- | :--- | :--- | :--- |
| **API Layer** | Spark Core API (Low-Level) | High-Level Relational API | High-Level Type-Safe API |
| **Language Support** | Python, Scala, Java, R | Python, Scala, Java, R | **Scala and Java only** (Not supported in Python due to dynamic typing) |
| **Ease of Coding** | Verbose and harder to write/maintain | Easy, intuitive, and declarative | Easy, intuitive, and highly type-safe |
| **Schema Awareness** | No explicit schema (Raw objects) | Structured table-like schema | Strongly-typed object schema |
| **Caching Targets** | Can be cached only in memory | Cached dynamically across memory and disk | Cached dynamically across memory and disk |
| **Built-in Optimization**| **No optimization engine**; relies entirely on user-written code efficiency. | Highly optimized via the **Catalyst Optimizer** because you provide structural schema metadata. | Highly optimized via the **Catalyst Optimizer** and fast Encoders. |


##### Cache vs. Persist: The Core Differences

| Feature | `cache()` | `persist()` |
| :--- | :--- | :--- |
| **Storage Level** | Fixed/Default storage level only. Cannot be customized. | **Fully Customizable** storage levels. |
| **DataFrame Default**| `MEMORY_AND_DISK` | `MEMORY_AND_DISK` (by default, if no level is specified) |
| **RDD Default** | `MEMORY_ONLY` | `MEMORY_ONLY` (by default, if no level is specified) |
| **Parameters** | Accepts no arguments. | Accepts a specific `StorageLevel` object. |

##### The 5 Dimensions of `persist()` Storage Levels
When choosing a custom storage level in `persist()`, you configure five internal flags:
1.  **useDisk:** Stores partitions on disk if they spill out of memory.
2.  **useMemory:** Keeps partitions cached in memory.
3.  **useOffHeap:** Stores data in off-heap memory (outside the standard JVM loop).
4.  **deserialized:** Keeps data deserialized in memory (faster to read, but uses significantly more RAM).
5.  **replication:** Defines the number of copies to make across different cluster nodes (e.g., `MEMORY_ONLY_2` keeps 2 replicas for high fault tolerance).

<img src="cache1.png" width="30%">

##### Critical Rules of Caching

*   **Lazy Evaluation:** Caching is a **lazy operation**. Calling `.cache()` neither triggers a transformation nor an action. It simply flags the DataFrame to be saved the next time an action runs.
*   **Size Warning:** Only cache DataFrames that will be reused multiple times across your pipeline. **Do not cache oversized datasets** that will choke cluster memory.
*   **Automatic Overflow:** By default on DataFrames (`MEMORY_AND_DISK`), if you try to cache a 10 GB dataset but only have 8 GB of available executor memory, Spark will cleanly save 8 GB in RAM and automatically spill the remaining 2 GB to local disk.
*   **Best Practice:** Always assign your cached result to a new variable or explicit dataframe reference:
    ```python
    order_cached_df = order_df.cache()
    ```

##### Action Warnings: Partial Caching Pitfalls
Not all actions cache your data equally. Spark optimizes execution paths, meaning operations that request fewer records will only cache a **partial fraction** of your dataset.

| Action Command | Caching Behavior | Why? |
| :--- | :--- | :--- |
| **`df.count()`** | **All partitions** are cached. | To calculate an accurate count, Spark must scan and evaluate every single partition. |
| **`df.show(20)`** | **Only 1 partition** is cached. | Spark only reads enough partitions to fulfill the top 20 rows requested. |
| **`df.head()`** | **Only 1 partition** is cached. | It stops processing as soon as it pulls the very first record. |
| **`df.tail(10)`**| **Only 1 partition** is cached. | It only scans the final partition to gather the trailing n records. |

##### Lifecycle & Performance Metrics

##### Unpersisting Data
Once your script finishes using a cached dataset, manually clear up cluster memory by dropping it:
```python
order_cached_df.unpersist()
```
##### Data Locality Levels
When data is cached cleanly, it updates Spark’s internal data locality schedule for future jobs:
*   **PROCESS_LOCAL:** The absolute fastest. The data lives cached directly inside the same JVM process running the task.
*   **NODE_LOCAL:** The data is stored on the local disk of the exact same machine running the job (requires minor disk I/O, but avoids network hops).

Serialization: is a process of converting objects into bytes that can be stored, transmitted across network 

In [6]:
from pyspark.storagelevel import StorageLevel
df.persist(StorageLevel(True, True, False, False, 1))

DataFrame[id: int, name: string, dob: date]

#### Parquet vs. Delta Lake: Feature Comparison

| Feature | Apache Parquet | Delta Lake |
| :--- | :--- | :--- |
| **Storage Architecture** | Columnar storage format | Columnar storage format (built *on top* of Parquet) |
| **Partitioning** | ✅ Supported | ✅ Supported |
| **Data Skipping** | ⚠️ Limited (relies on file-level metadata stats) | ✅ Highly Optimized (via transaction log data stats) |
| **Schema Enforcement** | ❌ No (can lead to corrupt reads if data mismatched) | ✅ Strict (prevents bad data insertion out-of-the-box) |
| **Schema Evolution** | ❌ Manual/Complicated | ✅ Built-in (allows changes over time safely) |
| **File Compaction** | ❌ Manual implementation required | ✅ Automated / Built-in (`OPTIMIZE` command) |
| **ACID Compliance** | ❌ No (no atomic commits or transaction safeguards) | ✅ Full ACID transactions via a metadata log |
| **Upserts & Merges** | ❌ No (requires a complete rewrite of partitions) | ✅ Full `MERGE`, `UPDATE`, and `DELETE` support |

##### Core Strengths of Delta Lake

*   **Metadata Log:** Every modification is recorded in an atomic `_delta_log` transaction ledger, providing ACID guarantees and enabling "Time Travel" version history.
*   **Data Skipping & Predicate Pushdown:** Leverages indexed file statistics inside the log to skip entirely irrelevant files during queries without opening them.
*   **Z-Ordering:** A multi-dimensional clustering technique that co-locates related data items in the same files, supercharging data skipping capabilities.
*   **Compaction:** Resolves the "small file problem" by automatically or manually merging smaller operational storage fragments into standard, readable large files.
*   **Optimized Writes:** Intelligently buffers and organizes file chunks right before writing them to the disk storage layer to prevent small file clutter.

##### Handling Schema Evolution in Delta Lake

When your incoming DataFrame structure expands or alters over time, you can instruct Delta Lake to safely evolve your table's layout instead of throwing an explicit schema validation crash.

##### Approach A: In-Line Write Configuration
Specify the `mergeSchema` option directly inside your writing pipeline block:

```python
new_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(delta_path)
```

##### Approach B: Global Session Setting
Alternatively, enable automatic merging globally across your entire Spark application runtime environment using configuration settings:

```python
# Enable schema evolution globally for the current session
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
```

##### Deep Dive: Why Data Skipping is Limited in Parquet vs. Delta Lake

To understand why data skipping is limited in standard Parquet, let's look at what happens behind the scenes during a simple filtering query:

```sql
SELECT * FROM table WHERE date = '2026-01-01';
```

##### 📦 The Parquet Architecture: Decentralized & Blind Scan
In a standard Parquet setup, Spark lacks a unified index or central coordinator. Finding relevant files requires a lot of heavy lifting:

1.  **File Listing:** Spark must issue a file listing command to your storage system (like S3 or HDFS) to find every single file in the directory.
2.  **Opening Footers:** Spark must open and read the **footer** of *every individual file* because Parquet stores its metadata (like min/max row group stats) at the end of each separate file.
3.  **Inconsistent Stats:** Min/max values are not guaranteed to be present. It depends entirely on the library and configuration used to write the files. If stats are missing, Spark is forced to blindly scan the entire dataset.
4.  **No Version Awareness:** Parquet cannot track which files are the newest, which were updated, or which can be safely ignored.


##### 🌟 The Delta Lake Architecture: Centralized Transaction Logs
Delta Lake overcomes these limitations by keeping an organized, centralized ledger known as the **`_delta_log`**.

1.  **Centralized Lookup:** Spark reads a single, consolidated metadata log instead of reaching out to hundreds of individual file footers.
2.  **Instant Skipping:** The log pre-records structural metadata and precise min/max statistics for every single file. Spark instantly identifies exactly which files match the `WHERE` clause before touching the main data storage.
3.  **Compaction Boost:** When Delta Lake compacts smaller files into larger ones, it narrows down the number of individual file entries in the log and refines the min/max boundaries, making data skipping even more precise.

##### Standard Parquet Execution
```text
Query: "Find data for Jan"

File1.parquet ──> (No Central Log) ──> Spark opens footer to check min/max stats...
File2.parquet ──> (No Central Log) ──> Spark opens footer to check min/max stats...
File3.parquet ──> (No Central Log) ──> Spark opens footer to check min/max stats...

👉 Result: Spark has to scan and read everything to be sure.
```

##### Delta Lake Execution
```text
Query: "Find data for Jan"
               │
               ▼
┌──────────────────────────────┐
│          _delta_log          │ ──> Spark checks the central ledger first.
│  File1.parquet ──> date: Jan  │
│  File2.parquet ──> date: Feb  │
│  File3.parquet ──> date: Mar  │
└──────────────────────────────┘
               │
               ▼
👉 Result: Spark instantly reads ONLY File1.parquet and skips the rest.
```
#### Z-Ordering Optimization in Delta Lake

Z-Ordering groups similar data together across your storage layout so Spark can read significantly fewer files during query execution.

*   **How it Works:** Z-Ordering uses a mathematical concept called a **space-filling curve (Z-curve)**. It rearranges incoming rows during a write/optimize operation to ensure that similar values stay clustered together within the same physical files.
*   **When to Use It:** 
    *   On columns that are frequently used in query filters (`WHERE` clauses).
    *   On **high-cardinality** columns (columns with many unique values, like `user_id` or `order_id`).

##### Direct Comparison: Partitioning vs. Z-Ordering

| Feature / Metric | Partitioning | Z-Ordering |
| :--- | :--- | :--- |
| **Ideal Cardinality** | Best for **low-cardinality** columns (e.g., `Year`, `Country`, `Status`). | Best for **high-cardinality** columns (e.g., `user_id`, `order_id`). |
| **Storage Structure** | physically creates separate **sub-directories/folders** on disk. | **Reorders and clusters data rows** inside the actual data files. |
| **Limitation** | Over-partitioning creates a "small file problem" and degrades performance. | Can be applied to multiple columns simultaneously without adding directory bloat. |

#### Handling Null Values in PySpark

When reading messy data sources, you can handle missing or null values during the initial ingestion phase or fix them dynamically using DataFrame operations.

##### Step 1: Explicitly identify Null strings on data load
```python
# Treat the literal string "NULL" as an actual SQL null value during CSV reading
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "NULL") \
    .csv("path/to/data.csv")
```

##### Step 2: Fill null values with a fallback default
```python
# Replace null values in a targeted column with a default value (e.g., 0)
df_cleaned = df.fillna({"col1": 0})
```

#### How Spark Achieves Fault Tolerance

Spark achieves absolute fault tolerance across a distributed cluster using two core pillars:

*   **Lineage Graphs (DAG):** Spark tracks the complete history of transformations (the data lineage) used to build a DataFrame or RDD. If a worker node crashes and a data partition is lost, Spark does not need to restart the entire application. It simply references the lineage graph to **recompute only the missing partitions** from their original source data.
*   **Task Retries:** If an executor fails to finish a task due to transient network drops or node failures, the Driver automatically reschedules and **retries the failed task** on an alternative healthy executor node.


#### Spark Optimization
Spark optimization is not just tuning spark.shuffle.partition or adding more executors. Its about understanding which layer is the bottleneck. For example,
* Too much data to be scanned? Then optimize file layout, pruning and filters
* large shuffle? optimize joins, aggregations, partitions
* One task taking longer than others? Check on skews
* Too many tiny tasks? Fix partiton size

##### Master checklist
* Query design: filter early, project only required columns
  less efficient
  df = spark.read.parquet("path")
  filtered_df = df.filter(df.col1 == "value").select("col1", "col2")
  better way
  df = spark.read.parquet("path").select("col1", "col2")
  filtered_df = df.filter(df.col1 == "value")
  Parquet supports column pruning, so it wont read all columns, Catalyst will handle this even if you write first version. But better to write second version and verify physical plan
* Avoid unneccessary actions:
  df.count()
  df.write.parquet(path)
  it runs upstream plans twice, if count is for debugging then remove it from production code
* Avoid unneccessary distinct(), sort(), repartition(). Catalyst already eliminates and simplifies operation, but goal is to not to manually reproduce every optimizer rule   
* Data format and storage: Parquet/ORC, compression, partition pruning
* Joins: Broadcast, sort-merge, shuffle-hash
* Shuffle and partitioning: Partiton size, coaleasce, repartition
* Catalyst and AQE
* Memory: executor sizing, caching, GC overhead, serialization, spill
* Pyspark/Scala code: Avoid UDF, use built-in function, efficient transformations
* Cluster and scheduling: executor sizing, mmeory allocation, cores, concurrent jobs
* Advanced architecture: Bucketing, incremental processing, skew aware design
* Prefer reduceByKey() or aggregateByKey() over groupBy(). It reduces amount of data shuffled
* If ordering needed only within partitions, use sortWithinPartitions() instead of sort()
* Choosing the shuffle partition: SP = shuffle data / 256 MB
  If shuffle data is 200 GB
  SP = (200*1024)/256 = 800 shuffle partitions
* Broadcast df manually if required
df = big_df.join(F.broadcast(small_df), on="id") <br>
check physical plan before forcing broadcast hint

##### Handling Data Skew

Data skew occurs when a few partitions hold the vast majority of the data, causing a few "straggler" tasks to bottleneck your entire cluster. Use the following workflow to diagnose and resolve it.

#### Step 1: Diagnose Skew via DataFrame Aggregation
To check if a specific key is causing the skew, aggregate your dataset and rank the highest volume keys:

```python
import pyspark.sql.functions as F

# Identify the top 10 most skewed keys
key_counts = df.groupBy("customer_id") \
               .count() \
               .orderBy(F.col("count").desc())

key_counts.show(10)
```

#### Step 2: Diagnose Skew via the Spark UI
If your code is running slowly, open the **Spark UI** and drill down into the longest-running stage:
*   **Task Duration:** Check if a few tasks take minutes while the rest finish in seconds.
*   **Shuffle Read Size:** Look for an uneven distribution of data sizes across tasks.
*   **Shuffle Spill (Memory/Disk):** Check if specific executors are running out of RAM and spilling massive chunks of data to disk.

#### Step 3: Mitigation via Adaptive Query Execution (AQE)
Instead of manually salting keys, you can leverage Spark's **Adaptive Query Execution (AQE)** framework to automatically detect skewed join keys at runtime and split them into smaller, parallel sub-tasks.

```python
# Enable runtime skew mitigation parameters
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```


#### Spark handling Data Skew - Always try this first

In [38]:
spark.conf.set(
    "spark.sql.adaptive.enabled", "true"
)
spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled", "true"
)

#### Manual Salting

In [62]:
# Create Dataframes: 40 orders for the skewed customer C001 and 10 orders for 10 other customers
hot_customer_orders = [
    (f"O{i:03d}", "C001", 100 + i)
    for i in range(1, 41)
]
other_customer_orders = [
    (f"O{i:03d}", f"C{i:03d}", 100 + i)
    for i in range(41, 51)
]
orders_data = hot_customer_orders + other_customer_orders
print(orders_data)
orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "amount"]
)
customers_data = [
    ("C001", "Alice"),
    ("C041", "Bob"),
    ("C042", "Charlie"),
    ("C043", "David"),
    ("C044", "Emma"),
    ("C045", "Frank"),
    ("C046", "Grace"),
    ("C047", "Helen"),
    ("C048", "Ivan"),
    ("C049", "Jack"),
    ("C050", "Kate")
]
customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name"]
)
orders_df.show(2)
customers_df.show(2)

[('O001', 'C001', 101), ('O002', 'C001', 102), ('O003', 'C001', 103), ('O004', 'C001', 104), ('O005', 'C001', 105), ('O006', 'C001', 106), ('O007', 'C001', 107), ('O008', 'C001', 108), ('O009', 'C001', 109), ('O010', 'C001', 110), ('O011', 'C001', 111), ('O012', 'C001', 112), ('O013', 'C001', 113), ('O014', 'C001', 114), ('O015', 'C001', 115), ('O016', 'C001', 116), ('O017', 'C001', 117), ('O018', 'C001', 118), ('O019', 'C001', 119), ('O020', 'C001', 120), ('O021', 'C001', 121), ('O022', 'C001', 122), ('O023', 'C001', 123), ('O024', 'C001', 124), ('O025', 'C001', 125), ('O026', 'C001', 126), ('O027', 'C001', 127), ('O028', 'C001', 128), ('O029', 'C001', 129), ('O030', 'C001', 130), ('O031', 'C001', 131), ('O032', 'C001', 132), ('O033', 'C001', 133), ('O034', 'C001', 134), ('O035', 'C001', 135), ('O036', 'C001', 136), ('O037', 'C001', 137), ('O038', 'C001', 138), ('O039', 'C001', 139), ('O040', 'C001', 140), ('O041', 'C041', 141), ('O042', 'C042', 142), ('O043', 'C043', 143), ('O044', '

In [71]:
# STEP 1: Check for DataSkew: Orders per customer
orders_df.groupBy("customer_id").count().orderBy(F.col("count").desc(), F.col("customer_id").asc()).show(5)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C001|   40|
|       C041|    1|
|       C042|    1|
|       C043|    1|
|       C044|    1|
+-----------+-----+
only showing top 5 rows



In [72]:
normal_join_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

In [82]:
NUM_SALTS = 4 # Should be number of shuffle partions 

# STEP 2: Add salt to the orders DataFrame
orders_salted_df = orders_df.withColumn(
    "salt", F.when(orders_df.customer_id == 'C001', 
    F.pmod(
        F.xxhash64("order_id"),
        F.lit(NUM_SALTS)
    ).cast("int")
).otherwise(F.lit(0)))

print("order_df distribution after salting")
orders_salted_df.groupBy(orders_salted_df.customer_id, orders_salted_df.salt).count().show(5)

salt_values_df = spark.range(NUM_SALTS).select(
    F.col("id").cast("int").alias("salt")
)
# Replicate each customer row for each salt.
customers_expanded_df = customers_df.crossJoin(
    salt_values_df
)
# Instead of crossJoin, we can also use array and explode
# skewed_customers_salted_df = customers_df.withColumn(
#     "salt",
#     F.explode(
#         F.array(
#             *[F.lit(i) for i in range(NUM_SALTS)]
#         )
#     )
# )
# skewed_customers_salted_df = skewed_customers_salted_df.filter(
#     (F.col("customer_id") == "C001")
#     | (F.col("salt") == 0)
# )

print("4 salt values * 11 total rows in customer_df:", salt_values_df.count(), customers_df.count(), customers_expanded_df.count())

# C001 gets all 4 salt values - 0,1,2,3. Other customers retain only salt 0.
customers_salted_df = customers_expanded_df.filter(
    (F.col("customer_id") == "C001")
    | (F.col("salt") == 0)
)
print("Rows in original customer dimension:", customers_df.count())
print("Rows in salted customer dimension:", customers_salted_df.count())

# STEP 3: Salted join
salted_join_df = orders_salted_df.join(
    customers_salted_df,
    on=["customer_id", "salt"],
    how="inner"
)

print("Normal join count:", normal_join_df.count())
print("Salted join count:", salted_join_df.count())
salted_join_df.orderBy("order_id").show(50)

order_df distribution after salting
+-----------+----+-----+
|customer_id|salt|count|
+-----------+----+-----+
|       C001|   0|   13|
|       C001|   2|   11|
|       C001|   1|    9|
|       C001|   3|    7|
|       C042|   0|    1|
+-----------+----+-----+
only showing top 5 rows

4 salt values * 11 total rows in customer_df: 4 11 44
Rows in original customer dimension: 11
Rows in salted customer dimension: 14
Normal join count: 50
Salted join count: 50
+-----------+----+--------+------+-------------+
|customer_id|salt|order_id|amount|customer_name|
+-----------+----+--------+------+-------------+
|       C001|   2|    O001|   101|        Alice|
|       C001|   0|    O002|   102|        Alice|
|       C001|   2|    O003|   103|        Alice|
|       C001|   1|    O004|   104|        Alice|
|       C001|   1|    O005|   105|        Alice|
|       C001|   0|    O006|   106|        Alice|
|       C001|   3|    O007|   107|        Alice|
|       C001|   0|    O008|   108|        Alice|

##### Bucketing

In [15]:
import time
spark.conf.set(
    "spark.sql.sources.bucketing.enabled", "true"
)
spark.conf.set(
    "spark.sql.bucketing.coalesceBucketsInJoin.enabled", "false"
)

configs = [
    "spark.sql.sources.bucketing.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.enabled",
    "spark.sql.bucketing.coalesceBucketsInJoin.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.files.maxPartitionBytes",
]

for key in configs:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception as e:
        print(f"{key}: {e}")

employee_details1 = spark.range(1, 10000001).select(
    F.col("id").cast("int").alias("emp_id"),
    F.concat(F.lit("Employee_"), F.col("id")).alias("emp_name"),
    (F.col("id") * 10 + 30000).cast("int").alias("salary")
)

employee_details2 = spark.range(1, 10000001).select(
    F.col("id").cast("int").alias("emp_id"),
    F.when(F.col("id") % 4 == 0, "IT")
    .when(F.col("id") % 4 == 1, "HR")
    .when(F.col("id") % 4 == 2, "Finance")
    .otherwise("Sales")
    .alias("department")
)

employee_details1.write.format("parquet").mode("overwrite").save("s3a://netapp-data-sample/employee_details1/")
employee_details2.write.format("parquet").mode("overwrite").save("s3a://netapp-data-sample/employee_details2/")

spark.sparkContext.setJobGroup("read_join_no_bucketing", "Read, Join and Write Without bucketing")
start_time = time.time()
employee_details1 = spark.read.format("parquet").load("s3a://netapp-data-sample/employee_details1/")
employee_details2 = spark.read.format("parquet").load("s3a://netapp-data-sample/employee_details2/")

emp_joined = employee_details1.join(employee_details2, on="emp_id", how="inner")

end_time = time.time()
print(f"\nTime taken with no bucketing: {end_time - start_time} seconds\n")
print(f"Explain plan for employee_joined without bucketing:")
emp_joined.explain("formatted")
emp_joined.write.format("parquet").mode("overwrite").save("s3a://netapp-data-sample/employee_joined_no_bucketing/")

spark.sparkContext.setJobGroup("dont_consider", "Dont consider this job group for metrics collection")
employee_details1.write \
    .format("parquet") \
    .mode("overwrite") \
    .bucketBy(4, "emp_id") \
    .saveAsTable("employee_bucketed1")

employee_details2.write \
    .format("parquet") \
    .mode("overwrite") \
    .bucketBy(4, "emp_id") \
    .saveAsTable("employee_bucketed2")

spark.sparkContext.setJobGroup("read_join_with_bucketing", "Read, Join and Write With bucketing")
start_time = time.time()

employee_details1 = spark.table("employee_bucketed1")
employee_details2 = spark.table("employee_bucketed2")

emp_joined = employee_details1.join(employee_details2, on="emp_id", how="inner")

end_time = time.time()
print(f"Time taken with bucketing: {end_time - start_time} seconds\n")
print(f"Explain plan for employee_joined with bucketing:")
emp_joined.explain("formatted")
emp_joined.write.format("parquet").mode("overwrite").save("s3a://netapp-data-sample/employee_joined_bucketed/")


spark.sql.sources.bucketing.enabled = true
spark.sql.autoBroadcastJoinThreshold = 10485760b
spark.sql.adaptive.autoBroadcastJoinThreshold = None
spark.sql.adaptive.enabled = true
spark.sql.bucketing.coalesceBucketsInJoin.enabled = false
spark.sql.shuffle.partitions = 200
spark.sql.files.maxPartitionBytes = 134217728b

Time taken with no bucketing: 4.664367198944092 seconds

Explain plan for employee_joined without bucketing:
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan parquet  (5)


(1) Scan parquet 
Output [3]: [emp_id#355, emp_name#356, salary#357]
Batched: true
Location: InMemoryFileIndex [s3a://netapp-data-sample/employee_details1]
PushedFilters: [IsNotNull(emp_id)]
ReadSchema: struct<emp_id:int,emp_name:string,salary:int>

(2) Filter
Input [3]

##### If you have bucketed the data it doesn't mean that spark will use bucket, it might go for broadcast join depending on the optimized plan. Always check for physical plan 

#### lit() 
df.withColumn("country", lit("India")) it creates a literal string value that spark can use as columnar expression <br>
df.withColumn("effective_start_date", current_timestamp()) current_timestamp() works without lit because current_timestamp() already returns column object <br>
df.filter(df.country == "India") Here lit is not needed because internally it will be converted to lit <br>
df.filter(df.country == lit("India")) It is also valid <br>
It is used when constant is be added as a columnar expression  

#### SCD Type1 and Type 2

In [223]:
# Incremental source set up for SCD type 1 and 2
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

# data = [(1, "A", "Blore", "a@gmail.com", "2026-10-18 19:59:00"), (2, "B", "HYD", "b@gmail.com", "2026-10-18 19:59:00"), (3, "C", "Chennai", "c@gmail.com", "2026-10-18 19:59:00")]
data = [(1, "A", "HYD", "a@gmail.com", "2026-10-18 20:59:00"), (4, "D", "HYD", "b@gmail.com", "2026-10-18 20:59:00")]
schema = StructType([StructField("id", IntegerType()), StructField("name", StringType()), StructField("city", StringType()), StructField("email", StringType()), StructField("updated_at", StringType())])
source_df = spark.createDataFrame(data, schema)
source_df = source_df.withColumn("source_updated_at", F.col("updated_at").cast(TimestampType())).drop("updated_at")
source_df.printSchema()

# TODO: deduplicate
# Handle deleted rows 

# Get last successful run from pipeline_status_table
# Connect to DB and read the last successful run time 
try: 
    print("Read from pipeline_status_table where pipeline = this and status = 'SUCCESS'")
    # last_successful_run = "1900-01-01" # For first load
    last_successful_run = "2026-10-18 19:59:00"
except Exception as e:
    # For the first run
    last_successful_run = "1900-01-01"
source_df = source_df.filter(F.col("source_updated_at") >= F.lit(last_successful_run).cast(TimestampType()))
source_df.show()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)

Read from pipeline_status_table where pipeline = this and status = 'SUCCESS'
+---+----+----+-----------+-------------------+
| id|name|city|      email|  source_updated_at|
+---+----+----+-----------+-------------------+
|  1|   A| HYD|a@gmail.com|2026-10-18 20:59:00|
|  4|   D| HYD|b@gmail.com|2026-10-18 20:59:00|
+---+----+----+-----------+-------------------+



#### SCD Type1 Incremental
##### Method 1: Overwrite all - applicable only if source is ALWAYS FULL LOAD
###### Increases compute cost 
###### Unneccessarily overwrite eveything if changes are very small and data volume is huge
###### Method 2: Using DeltaMerge

In [224]:
# Load the target
from delta.tables import DeltaTable

if last_successful_run == "1900-01-01":  # First run
    source_df.write.format("delta").partitionBy("city").save("s3a://netapp-data-sample/scd_1_target")
else:
    target_df = DeltaTable.forPath(spark, "s3a://netapp-data-sample/scd_1_target")
    # Update specific columns
    # target_df.alias("target").merge(source_df.alias("source"), "target.id = source.id")\
    #     .whenMatchedUpdate(set={"target.email" : "source.email",
    #                             "target.city": "source.city"})\
    #     .whenNotMatchedInsert(values={"target.id":"source.id",
    #                                   "target.name":"source.name",
    #                                   "target.city":"source.city",
    #                                   "target.email":"source.email",
    #                                   "target.updated_at":"source.updated_at"}).execute()
    # Update All columns
    target_df.alias("target").merge(source_df.alias("source"), "target.id = source.id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()  

target_delta = DeltaTable.forPath(spark, "s3a://netapp-data-sample/scd_1_target")
target_df = target_delta.toDF()
target_df.orderBy(target_df.id).show()

+---+----+-------+-----------+-------------------+
| id|name|   city|      email|  source_updated_at|
+---+----+-------+-----------+-------------------+
|  1|   A|    HYD|a@gmail.com|2026-10-18 20:59:00|
|  2|   B|    HYD|b@gmail.com|2026-10-18 19:59:00|
|  3|   C|Chennai|c@gmail.com|2026-10-18 19:59:00|
|  4|   D|    HYD|b@gmail.com|2026-10-18 20:59:00|
+---+----+-------+-----------+-------------------+



#### SCD Type2 Incremental

In [225]:
source_df = source_df.withColumn("effective_start_date", current_timestamp())\
    .withColumn("effective_end_date", lit("9999-12-31 00:00:00").cast(TimestampType()))\
    .withColumn("is_current", lit(True))

In [226]:
if last_successful_run == "1900-01-01":  # First run
    source_df.write.format("delta").partitionBy("city").save("s3a://netapp-data-sample/scd_2_target")
else:
    target_delta = DeltaTable.forPath(spark, "s3a://netapp-data-sample/scd_2_target")
    target_delta.alias("target").merge(source_df.alias("source"), "target.id = source.id AND target.is_current = 'true'")\
        .whenMatchedUpdate(set={"target.effective_end_date": "current_date()",
                                "target.is_current": "False"}).execute()
    target_delta.alias("target").merge(source_df.alias("source"), "target.id = source.id AND target.is_current = 'true'")\
    .whenNotMatchedInsertAll().execute()

verify_delta = DeltaTable.forPath(spark, "s3a://netapp-data-sample/scd_2_target")
verify_df = verify_delta.toDF()
verify_df.orderBy("id").show()

+---+----+-------+-----------+-------------------+--------------------+-------------------+----------+
| id|name|   city|      email|  source_updated_at|effective_start_date| effective_end_date|is_current|
+---+----+-------+-----------+-------------------+--------------------+-------------------+----------+
|  1|   A|    HYD|a@gmail.com|2026-10-18 20:59:00|2026-10-22 14:42:...|9999-12-31 00:00:00|      true|
|  1|   A|  Blore|a@gmail.com|2026-10-18 19:59:00|2026-10-22 14:40:...|2026-10-22 00:00:00|     false|
|  2|   B|    HYD|b@gmail.com|2026-10-18 19:59:00|2026-10-22 14:40:...|9999-12-31 00:00:00|      true|
|  3|   C|Chennai|c@gmail.com|2026-10-18 19:59:00|2026-10-22 14:40:...|9999-12-31 00:00:00|      true|
|  4|   D|    HYD|b@gmail.com|2026-10-18 20:59:00|2026-10-22 14:42:...|9999-12-31 00:00:00|      true|
+---+----+-------+-----------+-------------------+--------------------+-------------------+----------+



In [29]:
data = [
    (1, "Electronics", "Laptop", 5, 800),
    (2, "Electronics", "Mobile", 10, 500),
    (3, "Electronics", "Tablet", 8, 300),
    (4, "Electronics", "Headphones", 20, 50),
    (5, "Electronics", "Monitor", 6, 200),
    (6, "Electronics", "Laptop", 3, 800),
    (7, "Electronics", "Mobile", 4, 500),
    (8, "Electronics", "Tablet", 5, 300),
    (9, "Electronics", "Camera", 7, 400),
    (10, "Electronics", "Headphones", 10, 50),
    (11, "Furniture", "Chair", 20, 75),
    (12, "Furniture", "Table", 5, 300),
    (13, "Furniture", "Sofa", 4, 700),
    (14, "Furniture", "Bed", 3, 900),
    (15, "Furniture", "Desk", 8, 250),
    (16, "Furniture", "Chair", 10, 75),
    (17, "Furniture", "Table", 4, 300),
    (18, "Furniture", "Sofa", 2, 700),
    (19, "Furniture", "Bookshelf", 6, 150),
    (20, "Furniture", "Desk", 5, 250),
    (21, "Clothing", "T-Shirt", 50, 15),
    (22, "Clothing", "Jeans", 20, 40),
    (23, "Clothing", "Jacket", 10, 100),
    (24, "Clothing", "Shoes", 15, 80),
    (25, "Clothing", "Watch", 8, 120),
    (26, "Clothing", "T-Shirt", 30, 15),
    (27, "Clothing", "Jeans", 10, 40),
    (28, "Clothing", "Jacket", 5, 100),
    (29, "Clothing", "Shoes", 10, 80),
    (30, "Clothing", "Watch", 4, 120)
]

columns = [
    "sale_id",
    "category",
    "product",
    "quantity",
    "unit_price"
]

df = spark.createDataFrame(data, columns)
df.show()

+-------+-----------+----------+--------+----------+
|sale_id|   category|   product|quantity|unit_price|
+-------+-----------+----------+--------+----------+
|      1|Electronics|    Laptop|       5|       800|
|      2|Electronics|    Mobile|      10|       500|
|      3|Electronics|    Tablet|       8|       300|
|      4|Electronics|Headphones|      20|        50|
|      5|Electronics|   Monitor|       6|       200|
|      6|Electronics|    Laptop|       3|       800|
|      7|Electronics|    Mobile|       4|       500|
|      8|Electronics|    Tablet|       5|       300|
|      9|Electronics|    Camera|       7|       400|
|     10|Electronics|Headphones|      10|        50|
|     11|  Furniture|     Chair|      20|        75|
|     12|  Furniture|     Table|       5|       300|
|     13|  Furniture|      Sofa|       4|       700|
|     14|  Furniture|       Bed|       3|       900|
|     15|  Furniture|      Desk|       8|       250|
|     16|  Furniture|     Chair|      10|     

In [30]:
from pyspark.sql import functions as F
df = df.withColumn("sales", df.quantity*df.unit_price)
grp_df = df.groupBy(df.category, df.product).agg(sum(df.sales).alias("total_sales"))
w = Window.partitionBy(grp_df.category).orderBy(F.col("total_sales").desc())
w_df = grp_df.withColumn("rn", row_number().over(w))
w_df.show()

+-----------+----------+-----------+---+
|   category|   product|total_sales| rn|
+-----------+----------+-----------+---+
|   Clothing|     Shoes|       2000|  1|
|   Clothing|    Jacket|       1500|  2|
|   Clothing|     Watch|       1440|  3|
|   Clothing|     Jeans|       1200|  4|
|   Clothing|   T-Shirt|       1200|  5|
|Electronics|    Mobile|       7000|  1|
|Electronics|    Laptop|       6400|  2|
|Electronics|    Tablet|       3900|  3|
|Electronics|    Camera|       2800|  4|
|Electronics|Headphones|       1500|  5|
|Electronics|   Monitor|       1200|  6|
|  Furniture|      Sofa|       4200|  1|
|  Furniture|      Desk|       3250|  2|
|  Furniture|     Table|       2700|  3|
|  Furniture|       Bed|       2700|  4|
|  Furniture|     Chair|       2250|  5|
|  Furniture| Bookshelf|        900|  6|
+-----------+----------+-----------+---+



#### Some important Spark configs

In [113]:
configs = [
    "spark.databricks.delta.schema.autoMerge.enabled",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.bucketing.coalesceBucketsInJoin.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.files.maxPartitionNum",
    "spark.sql.files.minPartitionNum",
    "spark.sql.files.openCostInBytes",
    "spark.sql.shuffle.partitions",
    "spark.sql.sources.bucketing.autoBucketedScan.enabled",
    "spark.sql.sources.bucketing.enabled",
    # spark.dynamicAllocation.enabled
]

for key in configs:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception as e:
        print(f"{key}: {e}")
print("Default parallelism:", spark.sparkContext.defaultParallelism)

spark.databricks.delta.schema.autoMerge.enabled = false
spark.sql.adaptive.autoBroadcastJoinThreshold = None
spark.sql.adaptive.enabled = true
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5.0
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 268435456b
spark.sql.autoBroadcastJoinThreshold = 10485760b
spark.sql.bucketing.coalesceBucketsInJoin.enabled = false
spark.sql.files.maxPartitionBytes = 134217728b
spark.sql.files.maxPartitionBytes = 134217728b
spark.sql.files.maxPartitionBytes = 134217728b
spark.sql.files.maxPartitionBytes = 134217728b
spark.sql.files.maxPartitionNum = None
spark.sql.files.minPartitionNum = None
spark.sql.files.minPartitionNum = None
spark.sql.files.openCostInBytes = 4194304b
spark.sql.files.openCostInBytes = 4194304b
spark.sql.shuffle.partitions = 200
spark.sql.sources.bucketing.autoBucketedScan.enabled = true
spark.sql.sources.bucketing.enabled = true
Default parallelism: 16


In [115]:
df = spark.read.format("parquet").load("s3a://netapp-data-sample/tlc/fhvhv_tripdata_2025-01.parquet")
print(df.rdd.getNumPartitions())
print(spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("Here, file size is 468.3 MB, but number of partitions is 16, not 4 (468/128) though \
maxPartitionBytes is 128 MB, because Default parallelism is 16")
# To change number of partitions 
# spark.conf.set("spark.sql.files.minPartitionNum", 4)
# spark.conf.unset("spark.sql.files.minPartitionNum") unset

16
134217728b
Here, file size is 468.3 MB, but number of partitions is 16, not 4 (468/128) though maxPartitionBytes is 128 MB, because Default parallelism is 16


In [120]:
partition_sizes = df.withColumn("_partition_id", F.spark_partition_id()).groupBy("_partition_id").count().orderBy("_partition_id")
partition_sizes.show(10)

+-------------+-------+
|_partition_id|  count|
+-------------+-------+
|            0|1048576|
|            1|1048576|
|            2|2097152|
|            3|1048576|
|            4|1048576|
|            5|1048576|
|            6|2097152|
|            7|1048576|
|            8|1048576|
|            9|1048576|
+-------------+-------+
only showing top 10 rows



#### Deciding and Tuning Spark cluster configuration
Data size: 100GB 
File type: Parquet file
Transformations: Joining, groupBy, Aggregations, filters, derived columns etc
Start with REASONABLE BASELINE
8 worker nodes, 64 GB RAM, 2 executors per worker, 16 vCPUs per worker node
= 16 executors, 512 GB RAM (64*8), 128 vCPUs (16*8)

16 GB RAM per executor
4 cores per executor = 16*4 = 64 concurrent tasks 

<img src="spark_conf.png">

# 🛠️ Spark Troubleshooting & Error Resolution Guide

## 1. `spark.executor.memoryOverhead` Issues

### 💡 What is it?
* **Off-Heap Memory:** Added on top of `spark.executor.memory`.
* **Usage:** Allocated for native libraries, large shuffles, and Python UDFs.

### 🚨 Error Messages
* `Container killed by YARN for exceeding memory limits`
* `Exit code 137` or `Exit code 143`
* Random executor losses

### 🔍 Causes
* Heavy usage of Python/Custom UDFs.
* Extremely large `groupBy` or `join` operations.
* Large **broadcast variables** referenced inside Python UDFs.

### 🛠️ Solutions
* **Adjust Overhead:** Increase overhead memory:
  ```properties
  spark.executor.memoryOverhead = max(10-20% of executor memory, 2-4GB)
  ```
* **Optimize Shuffles:** Target 128 MB to 256 MB partition sizes:
  ```properties
  spark.sql.shuffle.partitions = (Total Shuffle Data Size / 200MB)
  ```
* **Code Fixes:** 
  * Fix data skew and avoid using `.collect()`.
  * Prefer built-in PySpark functions over custom Python UDFs wherever possible.

## 2. `spark.executor.memory` (Heap) Issues

### 🚨 Error Messages
* `java.lang.OutOfMemoryError: Java heap space`
* `GC overhead limit exceeded`

### 🔍 Causes
* Severe **data skew** (a single executor handling the bulk of the data processing).

### 🛠️ Solutions
* Implement data-skew mitigation techniques (e.g., salting, filtering null keys before joins).
* Scale up the heap memory configuration:
  ```properties
  spark.executor.memory
  ```

## 3. Long Shuffle Stages & Task Hangs

### 🚨 Symptoms
* Some tasks take forever to finish while others complete instantly.
* Huge amounts of data spilling over to disk (`Spill (Disk)` visible in Spark UI).

### 🔍 Causes
* Severe data skew on join/grouping keys.

### 🛠️ Solutions
* Identify and fix the skewed keys.
* Fine-tune `spark.sql.shuffle.partitions` to redistribute data load across more tasks.

## 4. App Fails Suddenly (Previously Working Fine)

### 🔍 Causes & Diagnosis
* **Data Mutations:** Incoming data structure changed, encountered unexpected `null` values, or triggered unhandled schema evolution.
* **Dependencies:** Spark version upgrades or breaking changes from underlying JAR/library updates.
* **Infrastructure:** Target storage full, transient network partitions, or storage throttling.
* **Security & Auth:** Revoked access tokens, expired credentials, or updated User Group policies.

## 5. Job Suddenly Becomes Slower

### 🔍 Causes
* Unanticipated increase in incoming data volume.
* Data size surpassed the `spark.sql.autoBroadcastJoinThreshold`, reverting to a heavy SortMergeJoin.
* Data skew or a **shuffle explosion** caused by structural changes in join conditions (e.g., cross joins).

## 6. Performance Drops (With Constant Data Size)

### 🧭 Debugging Framework
1. Isolate the exact **Stage** causing the bottleneck using the Spark UI.
2. Compare the failing execution metrics against a **historical good run** using:
   * Spark UI event timelines
   * Driver/Executor log outputs
   * Catalyst physical execution plans (`.explain(true)`)
   * Cluster-level utilization metrics (CPU/Memory/IO)

### 🔍 Root Causes
* **Hidden Skew:** Total data size remains identical, but a few specific partitions contain significantly more rows than others.
* **Plan Regression:** Catalyst Optimizer changed its approach (e.g., dropped partition pruning optimizations or swapped join strategies).
* **Resource Contention:** Fewer active cluster executors available, unhealthy nodes, or noisy neighbours running concurrent jobs.
* **Downstream Bottlenecks:** External write targets (databases, data lakes, APIs) experiencing high latency or structural slow-downs.


### 📝 TODO: 

Z order demo <br>
data cleaning <br>
data validation <br>
data quality checks <br>
regex <br>
parity checks <br>
null handling <br>
schema evolution <br>

Caching Spark Tables
To cache a structured catalog table directly via Spark SQL instead of a DataFrame variable, use:
```sql
CACHE TABLE my_spark_table;
-- To release it:
UNCACHE TABLE my_spark_table;
```

### next project: CDC - 20th Oct